# CheXpert 6 nhãn — Multi-task + Relation-aware learning cho Pneumonia và Pneumothorax

Notebook này là **bản thực nghiệm** mở rộng từ pipeline 6 nhãn trước đó. Mục tiêu là kiểm tra xem hai nhãn có AUROC thấp hơn — **Pneumonia** và **Pneumothorax** — có cải thiện khi mô hình học thêm các nhãn liên quan hay không.

## Ý tưởng chính

**Nhãn chính cần báo cáo:**

- Atelectasis
- Cardiomegaly
- Edema
- Pleural Effusion
- Pneumonia
- Pneumothorax

**Nhãn phụ (auxiliary tasks):**

- Lung Opacity
- Consolidation
- Pleural Other

**Quan hệ thử nghiệm:**

- `Pneumonia` nhận thêm tín hiệu dự đoán từ `Lung Opacity`, `Consolidation`, `Atelectasis`, `Edema`.
- `Pneumothorax` nhận thêm tín hiệu dự đoán từ `Pleural Effusion`, `Pleural Other`.

Mô hình **không dùng ground-truth của nhãn liên quan khi inference**. Relation head chỉ nhận feature ảnh và **xác suất do chính mô hình dự đoán** cho các nhãn nguồn, vì vậy không có label leakage ở thời điểm suy luận.

> Đây là kiến trúc relation-aware để làm ablation, không phải khẳng định quan hệ nhân quả hay một hierarchy lâm sàng cứng. Hiệu quả phải được kiểm chứng bằng AUROC/AUPRC trên development/internal validation với cùng split.

## Thiết kế ablation đề nghị

| Experiment | Auxiliary tasks | Relation head | Pneumonia uncertainty | Pneumothorax uncertainty |
|---|---:|---:|---|---|
| `B0_main6` | Không | Không | U-Ignore | U-Ignore |
| `E1_aux_multitask` | Có | Không | U-Ignore | U-Ignore |
| `E2_relation_aware` | Có | Có | U-Ignore | U-Ignore |
| `E3_relation_aware_lsr` | Có | Có | LSR-positive | LSR-positive |

Giữ nguyên seed, patient-level split, backbone, optimizer, số epoch và preprocessing để so sánh công bằng.

In [ ]:
# 1. Imports + seed
import os
import random
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_curve,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE =', DEVICE)

## 2. Cấu hình thí nghiệm

Mặc định notebook chạy `E2_relation_aware`, tức dùng 3 auxiliary labels và hai relation heads nhưng vẫn giữ `U-Ignore` cho Pneumonia/Pneumothorax để tách riêng tác động của **quan hệ nhãn** khỏi tác động của **label smoothing**.

Sau khi có kết quả E2, đổi `EXPERIMENT_NAME` sang E3 để kiểm tra thêm LSR.

In [ ]:
# 2. Config
DATA_ROOT = Path('/kaggle/input/datasets/ashery/chexpert')
TRAIN_CSV = DATA_ROOT / 'train.csv'
OFFICIAL_VALID_CSV = DATA_ROOT / 'valid.csv'

MAIN_LABELS = [
    'Atelectasis',
    'Cardiomegaly',
    'Edema',
    'Pleural Effusion',
    'Pneumonia',
    'Pneumothorax',
]

AUX_LABELS = [
    'Lung Opacity',
    'Consolidation',
    'Pleural Other',
]

# Relation targets chỉ dùng predicted probabilities của các source labels.
RELATION_SOURCES = {
    'Pneumonia': ['Lung Opacity', 'Consolidation', 'Atelectasis', 'Edema'],
    'Pneumothorax': ['Pleural Effusion', 'Pleural Other'],
}

EXPERIMENTS = {
    'B0_main6': {
        'use_aux': False,
        'use_relations': False,
        'pneumonia_policy': 'u_ignore',
        'pneumothorax_policy': 'u_ignore',
    },
    'E1_aux_multitask': {
        'use_aux': True,
        'use_relations': False,
        'pneumonia_policy': 'u_ignore',
        'pneumothorax_policy': 'u_ignore',
    },
    'E2_relation_aware': {
        'use_aux': True,
        'use_relations': True,
        'pneumonia_policy': 'u_ignore',
        'pneumothorax_policy': 'u_ignore',
    },
    'E3_relation_aware_lsr': {
        'use_aux': True,
        'use_relations': True,
        'pneumonia_policy': 'lsr_pos',
        'pneumothorax_policy': 'lsr_pos',
    },
}

EXPERIMENT_NAME = 'E2_relation_aware'
CFG = EXPERIMENTS[EXPERIMENT_NAME]

# Policy 4 nhãn chính giữ giống pipeline trước.
MAIN_POLICIES = {
    'Atelectasis': 'u_ones',
    'Cardiomegaly': 'u_multiclass',
    'Edema': 'u_ones',
    'Pleural Effusion': 'u_multiclass',
    'Pneumonia': CFG['pneumonia_policy'],
    'Pneumothorax': CFG['pneumothorax_policy'],
}

# Auxiliary labels: mặc định U-Ignore để không áp đặt hard mapping cho uncertain.
AUX_POLICIES = {
    'Lung Opacity': 'u_ignore',
    'Consolidation': 'u_ignore',
    'Pleural Other': 'u_ignore',
}

ACTIVE_LABELS = MAIN_LABELS + (AUX_LABELS if CFG['use_aux'] else [])
POLICIES = {**MAIN_POLICIES, **({k: AUX_POLICIES[k] for k in AUX_LABELS} if CFG['use_aux'] else {})}

# Dữ liệu / train
IMG_SIZE = 320
BATCH_SIZE = 16
NUM_WORKERS = 4
EPOCHS = 5
LR = 1e-4
WEIGHT_DECAY = 1e-5

# Missing labels: giữ 'zero' như pipeline cũ để E1/E2 chỉ thay auxiliary/relation.
# Sau khi có baseline sạch, thử 'ignore' như một ablation riêng cho cách xử lý NaN.
NAN_TRAIN_POLICY = 'zero'  # 'zero' để so sánh công bằng với pipeline cũ; thử 'ignore' như ablation riêng

# Loss
LOSS_MODE = 'weighted_bce'   # 'weighted_bce' hoặc 'asl'
POS_WEIGHT_CAP = 10.0
ASL_GAMMA_NEG = 4.0
ASL_GAMMA_POS = 1.0
ASL_CLIP = 0.05
AUX_LOSS_WEIGHT = 0.35

# Relation head
RELATION_DETACH = False  # False: gradient từ target có thể cập nhật source heads
RELATION_HIDDEN = 256

# Evaluation / model selection
TARGET_RECALL = 0.85
CHECKPOINT_METRIC = 'macro_auroc_main'  # ưu tiên AUROC vì mục tiêu là cải thiện AUROC
TRAIN_RATIO = 0.80
DEV_RATIO = 0.10
VALID_RATIO = 0.10

OUT_DIR = Path('chexpert_relation_results')
OUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_PATH = OUT_DIR / f'best_{EXPERIMENT_NAME}_{LOSS_MODE}.pt'

print('Experiment:', EXPERIMENT_NAME)
print('Use aux:', CFG['use_aux'])
print('Use relations:', CFG['use_relations'])
print('Active labels:', ACTIVE_LABELS)
print('Policies:', POLICIES)
print('Checkpoint:', CHECKPOINT_PATH)

## 3. Đọc dữ liệu và chia theo bệnh nhân

Chỉ giữ ảnh frontal và chia `train / development / internal validation` theo `PatientID` để tránh cùng bệnh nhân xuất hiện ở nhiều split.

Official validation **không được dùng để chọn checkpoint hoặc threshold**. Nếu có `valid.csv`, notebook chỉ đánh giá nó ở cuối sau khi đã khóa cấu hình.

In [ ]:
# 3. Load CSV + split patient-level
assert TRAIN_CSV.exists(), f'Không tìm thấy: {TRAIN_CSV}'

df = pd.read_csv(TRAIN_CSV)
required_cols = ['Path', 'Frontal/Lateral', *ACTIVE_LABELS]
missing_cols = [c for c in required_cols if c not in df.columns]
assert not missing_cols, f'Thiếu cột: {missing_cols}'

frontal_df = df[df['Frontal/Lateral'].astype(str).str.lower().eq('frontal')].copy()
frontal_df['PatientID'] = frontal_df['Path'].str.extract(r'(patient\d+)', expand=False)
assert frontal_df['PatientID'].notna().all(), 'Không tách được PatientID từ một số Path.'

patients = np.array(sorted(frontal_df['PatientID'].unique()))
rng = np.random.default_rng(SEED)
rng.shuffle(patients)

n_patients = len(patients)
n_train = int(n_patients * TRAIN_RATIO)
n_dev = int(n_patients * DEV_RATIO)

train_patients = set(patients[:n_train])
dev_patients = set(patients[n_train:n_train+n_dev])
valid_patients = set(patients[n_train+n_dev:])

assert train_patients.isdisjoint(dev_patients)
assert train_patients.isdisjoint(valid_patients)
assert dev_patients.isdisjoint(valid_patients)

train_df = frontal_df[frontal_df['PatientID'].isin(train_patients)].reset_index(drop=True)
dev_df = frontal_df[frontal_df['PatientID'].isin(dev_patients)].reset_index(drop=True)
valid_df = frontal_df[frontal_df['PatientID'].isin(valid_patients)].reset_index(drop=True)

split_summary = pd.DataFrame({
    'Split': ['Train', 'Develop', 'Internal valid'],
    'Images': [len(train_df), len(dev_df), len(valid_df)],
    'Patients': [
        train_df['PatientID'].nunique(),
        dev_df['PatientID'].nunique(),
        valid_df['PatientID'].nunique(),
    ],
})
print(split_summary.to_string(index=False))

## 4. Kiểm tra phân bố nhãn và mối quan hệ trên train split

Cell dưới chỉ dùng các cặp nhãn raw `0/1` để tính thống kê quan hệ; `-1` và `NaN` bị bỏ khỏi phép tính. Mục đích là kiểm tra xem các source labels có thực sự cung cấp tín hiệu trên dữ liệu của bạn trước khi tin vào relation head.

In [ ]:
# 4. Raw label stats
def raw_label_stats(frame, labels):
    rows = []
    for label in labels:
        s = pd.to_numeric(frame[label], errors='coerce')
        rows.append({
            'Label': label,
            'Positive (1)': int((s == 1).sum()),
            'Negative (0)': int((s == 0).sum()),
            'Uncertain (-1)': int((s == -1).sum()),
            'NaN': int(s.isna().sum()),
        })
    return pd.DataFrame(rows)

stats_df = raw_label_stats(train_df, ACTIVE_LABELS)
stats_df

In [ ]:
# 5. Conditional prevalence cho các relation pairs
# Chỉ dùng rows mà cả target và source đều là 0/1.
def conditional_relation_table(frame, relation_sources):
    rows = []
    for target, sources in relation_sources.items():
        if target not in frame.columns:
            continue
        for source in sources:
            if source not in frame.columns:
                continue

            t = pd.to_numeric(frame[target], errors='coerce')
            s = pd.to_numeric(frame[source], errors='coerce')
            mask = t.isin([0, 1]) & s.isin([0, 1])
            tt = t[mask].astype(int)
            ss = s[mask].astype(int)

            n = int(mask.sum())
            n_s1 = int((ss == 1).sum())
            n_s0 = int((ss == 0).sum())
            p_t1_s1 = float(tt[ss == 1].mean()) if n_s1 else np.nan
            p_t1_s0 = float(tt[ss == 0].mean()) if n_s0 else np.nan

            rows.append({
                'Target': target,
                'Source': source,
                'N paired 0/1': n,
                'P(target=1 | source=1)': p_t1_s1,
                'P(target=1 | source=0)': p_t1_s0,
                'Risk ratio (descriptive)': (p_t1_s1 / p_t1_s0) if p_t1_s0 and not np.isnan(p_t1_s0) else np.nan,
            })
    return pd.DataFrame(rows)

relation_table = conditional_relation_table(train_df, RELATION_SOURCES)
relation_table.round(4)

## 5. Preprocessing

Giữ preprocessing gần pipeline trước để ablation tập trung vào thay đổi **auxiliary/relations**. Có thể bật augmentation nhẹ sau khi đã có baseline ổn định.

In [ ]:
# 6. Transforms
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    # Bật khi muốn làm ablation augmentation:
    # transforms.RandomAffine(degrees=5, translate=(0.02, 0.02), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
# 7. Resolve image path
def resolve_image_path(path_str: str) -> Path:
    raw = str(path_str).replace('\\', '/')
    p = Path(raw)

    candidates = [DATA_ROOT / p, DATA_ROOT.parent / p]
    parts = list(p.parts)

    for prefix in ('CheXpert-v1.0-small', 'CheXpert-v1.0'):
        if prefix in parts:
            idx = parts.index(prefix)
            candidates.append(DATA_ROOT / Path(*parts[idx+1:]))

    for anchor in ('train', 'valid', 'test'):
        if anchor in parts:
            idx = parts.index(anchor)
            candidates.append(DATA_ROOT / Path(*parts[idx:]))

    for c in candidates:
        if c.exists():
            return c

    raise FileNotFoundError(
        f'Không tìm thấy ảnh. CSV Path: {path_str}\n'
        f'DATA_ROOT: {DATA_ROOT}\nĐã thử:\n' + '\n'.join(str(c) for c in candidates)
    )

example_path = resolve_image_path(frontal_df.iloc[0]['Path'])
print('Example:', example_path)
print('Exists:', example_path.exists())

## 6. Dataset và uncertainty policies

- `u_ones`: `-1 -> 1`
- `u_zeros`: `-1 -> 0`
- `u_ignore`: bỏ `-1` khỏi loss
- `u_multiclass`: `0 / 1 / uncertain` là 3 lớp
- `lsr_pos`: `-1 -> Uniform(0.55, 0.85)`
- `lsr_neg`: `-1 -> Uniform(0.00, 0.30)`

Để so sánh công bằng với pipeline cũ, notebook mặc định `NaN -> 0` khi training (`NAN_TRAIN_POLICY='zero'`). Sau khi hoàn tất B0/E1/E2/E3, có thể chạy thêm một ablation riêng với `NAN_TRAIN_POLICY='ignore'`.

In [ ]:
# 8. Dataset
class CheXpertRelationDataset(Dataset):
    def __init__(self, frame, labels, policies, transform=None, training=False, nan_train_policy='ignore'):
        self.frame = frame.reset_index(drop=True)
        self.labels = labels
        self.policies = policies
        self.transform = transform
        self.training = training
        self.nan_train_policy = nan_train_policy

    def __len__(self):
        return len(self.frame)

    def _binary_target(self, raw, policy):
        if pd.isna(raw):
            if self.nan_train_policy == 'zero':
                return 0.0, 1.0
            return 0.0, 0.0

        raw = float(raw)
        if raw == 1.0:
            return 1.0, 1.0
        if raw == 0.0:
            return 0.0, 1.0
        if raw == -1.0:
            if policy == 'u_ones':
                return 1.0, 1.0
            if policy == 'u_zeros':
                return 0.0, 1.0
            if policy == 'u_ignore':
                return 0.0, 0.0
            if policy == 'lsr_pos':
                return float(np.random.uniform(0.55, 0.85)), 1.0
            if policy == 'lsr_neg':
                return float(np.random.uniform(0.00, 0.30)), 1.0
        raise ValueError(f'Không hỗ trợ raw={raw}, policy={policy}')

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        img_path = resolve_image_path(row['Path'])
        image = Image.open(img_path).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)

        binary_targets = {}
        binary_masks = {}
        multiclass_targets = {}
        multiclass_masks = {}
        raw_values = {}
        eval_masks = {}

        for label in self.labels:
            raw = pd.to_numeric(pd.Series([row[label]]), errors='coerce').iloc[0]
            raw_values[label] = np.nan if pd.isna(raw) else float(raw)
            eval_masks[label] = float((not pd.isna(raw)) and float(raw) in (0.0, 1.0))

            policy = self.policies[label]
            if policy == 'u_multiclass':
                if pd.isna(raw):
                    target = 0
                    m = 1.0 if self.nan_train_policy == 'zero' else 0.0
                elif float(raw) == 0.0:
                    target, m = 0, 1.0
                elif float(raw) == 1.0:
                    target, m = 1, 1.0
                elif float(raw) == -1.0:
                    target, m = 2, 1.0
                else:
                    raise ValueError(raw)
                multiclass_targets[label] = target
                multiclass_masks[label] = m
            else:
                target, m = self._binary_target(raw, policy)
                binary_targets[label] = target
                binary_masks[label] = m

        return {
            'image': image,
            'path': str(row['Path']),
            'patient_id': row.get('PatientID', ''),
            'binary_targets': {k: torch.tensor(v, dtype=torch.float32) for k, v in binary_targets.items()},
            'binary_masks': {k: torch.tensor(v, dtype=torch.float32) for k, v in binary_masks.items()},
            'multiclass_targets': {k: torch.tensor(v, dtype=torch.long) for k, v in multiclass_targets.items()},
            'multiclass_masks': {k: torch.tensor(v, dtype=torch.float32) for k, v in multiclass_masks.items()},
            'raw_values': {k: torch.tensor(0.0 if np.isnan(v) else v, dtype=torch.float32) for k, v in raw_values.items()},
            'eval_masks': {k: torch.tensor(v, dtype=torch.float32) for k, v in eval_masks.items()},
        }

In [ ]:
# 9. DataLoaders
train_ds = CheXpertRelationDataset(
    train_df, ACTIVE_LABELS, POLICIES, train_transform,
    training=True, nan_train_policy=NAN_TRAIN_POLICY
)
dev_ds = CheXpertRelationDataset(
    dev_df, ACTIVE_LABELS, POLICIES, eval_transform,
    training=False, nan_train_policy=NAN_TRAIN_POLICY
)
valid_ds = CheXpertRelationDataset(
    valid_df, ACTIVE_LABELS, POLICIES, eval_transform,
    training=False, nan_train_policy=NAN_TRAIN_POLICY
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
)
dev_loader = DataLoader(
    dev_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
valid_loader = DataLoader(
    valid_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

sample = train_ds[0]
print('image:', sample['image'].shape)
print('binary labels:', list(sample['binary_targets']))
print('multiclass labels:', list(sample['multiclass_targets']))

## 7. Pos-weight

Với `NAN_TRAIN_POLICY='zero'`, NaN được tính như negative để giữ đúng baseline cũ. Nếu đổi sang `'ignore'`, NaN sẽ bị loại khỏi cả loss và phép tính `pos_weight`. Với `u_ignore`, uncertain luôn bị loại khỏi loss.

In [ ]:
# 10. Binary/multiclass labels + pos weights
BINARY_LABELS = [l for l in ACTIVE_LABELS if POLICIES[l] != 'u_multiclass']
MULTICLASS_LABELS = [l for l in ACTIVE_LABELS if POLICIES[l] == 'u_multiclass']

def compute_binary_pos_weights(frame, labels, policies, nan_train_policy='ignore'):
    result = {}
    rows = []

    for label in labels:
        if policies[label] == 'u_multiclass':
            continue

        policy = policies[label]
        s = pd.to_numeric(frame[label], errors='coerce')

        if policy == 'u_ones':
            pos = int(((s == 1) | (s == -1)).sum())
            neg = int((s == 0).sum())
        elif policy == 'u_zeros':
            pos = int((s == 1).sum())
            neg = int(((s == 0) | (s == -1)).sum())
        elif policy == 'u_ignore':
            pos = int((s == 1).sum())
            neg = int((s == 0).sum())
        elif policy in ('lsr_pos', 'lsr_neg'):
            # Pos-weight dựa trên explicit 0/1; uncertain có soft target riêng trong loss.
            pos = int((s == 1).sum())
            neg = int((s == 0).sum())
        else:
            raise ValueError(policy)

        if nan_train_policy == 'zero':
            neg += int(s.isna().sum())

        raw_w = neg / max(pos, 1)
        used_w = min(raw_w, POS_WEIGHT_CAP)
        result[label] = used_w
        rows.append({
            'Label': label,
            'Policy': policy,
            'Pos count': pos,
            'Neg count': neg,
            'Raw pos_weight': raw_w,
            'Used pos_weight': used_w,
        })

    return result, pd.DataFrame(rows)

POS_WEIGHTS, pos_weight_table = compute_binary_pos_weights(
    train_df, ACTIVE_LABELS, POLICIES, NAN_TRAIN_POLICY
)
pos_weight_table

## 8. Relation-aware DenseNet121

Cách hoạt động:

1. DenseNet121 tạo shared image feature.
2. Mỗi active label có một `base head`.
3. Nếu bật relation learning:
   - `Pneumonia` lấy thêm predicted probabilities của `Lung Opacity`, `Consolidation`, `Atelectasis`, `Edema`.
   - `Pneumothorax` lấy thêm predicted probabilities của `Pleural Effusion`, `Pleural Other`.
4. Relation head dự đoán một **delta logit** và cộng vào base logit của target.

Vì source signals là **dự đoán của mô hình**, không phải ground-truth, cấu trúc này dùng được khi inference trên ảnh mới.

In [ ]:
# 11. Helpers chuyển logits -> positive probability
def positive_probability_from_logits(label, logits, policies):
    if policies[label] == 'u_multiclass':
        # Restricted softmax trên negative(0) và positive(1), bỏ uncertain(2)
        return torch.softmax(logits[:, :2], dim=1)[:, 1]
    return torch.sigmoid(logits.squeeze(1))


class RelationAwareDenseNet121(nn.Module):
    def __init__(
        self,
        active_labels,
        policies,
        relation_sources,
        use_relations=True,
        relation_hidden=256,
        relation_detach=False,
        pretrained=True,
    ):
        super().__init__()
        self.active_labels = list(active_labels)
        self.policies = policies
        self.use_relations = use_relations
        self.relation_detach = relation_detach

        weights = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        net = models.densenet121(weights=weights)
        in_features = net.classifier.in_features
        net.classifier = nn.Identity()
        self.backbone = net
        self.in_features = in_features

        self.base_heads = nn.ModuleDict()
        for label in self.active_labels:
            out_dim = 3 if policies[label] == 'u_multiclass' else 1
            self.base_heads[label] = nn.Linear(in_features, out_dim)

        # Chỉ tạo relation head khi target + source labels đều active.
        self.relation_sources = {}
        self.relation_heads = nn.ModuleDict()

        if self.use_relations:
            for target, sources in relation_sources.items():
                if target not in self.active_labels:
                    continue
                assert policies[target] != 'u_multiclass', f'{target} relation target phải là binary head.'
                valid_sources = [s for s in sources if s in self.active_labels]
                if not valid_sources:
                    continue
                self.relation_sources[target] = valid_sources

                # feature + base target logit + one probability per source
                rel_dim = in_features + 1 + len(valid_sources)
                self.relation_heads[target] = nn.Sequential(
                    nn.Linear(rel_dim, relation_hidden),
                    nn.ReLU(inplace=True),
                    nn.Dropout(p=0.2),
                    nn.Linear(relation_hidden, 1),
                )

    def forward(self, x):
        feat = self.backbone(x)
        base = {label: self.base_heads[label](feat) for label in self.active_labels}
        outputs = dict(base)

        if not self.use_relations:
            return outputs

        for target, sources in self.relation_sources.items():
            source_probs = []
            for source in sources:
                p = positive_probability_from_logits(source, base[source], self.policies)
                if self.relation_detach:
                    p = p.detach()
                source_probs.append(p.unsqueeze(1))

            base_target_logit = base[target].squeeze(1)
            relation_input = torch.cat(
                [feat, base_target_logit.unsqueeze(1), *source_probs], dim=1
            )
            delta = self.relation_heads[target](relation_input).squeeze(1)
            final_logit = base_target_logit + delta
            outputs[target] = final_logit.unsqueeze(1)

        return outputs

model = RelationAwareDenseNet121(
    active_labels=ACTIVE_LABELS,
    policies=POLICIES,
    relation_sources=RELATION_SOURCES,
    use_relations=CFG['use_relations'],
    relation_hidden=RELATION_HIDDEN,
    relation_detach=RELATION_DETACH,
    pretrained=True,
).to(DEVICE)

print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))
print('Relation sources used:', model.relation_sources)

## 9. Loss: main tasks + auxiliary tasks

Main labels có trọng số loss `1.0`; auxiliary labels mặc định `0.35`. Mục tiêu là để auxiliary tasks định hình shared representation mà không lấn át 6 task chính.

Có thể ablation `AUX_LOSS_WEIGHT` trong `{0.1, 0.25, 0.35, 0.5}` nếu E1/E2 không cải thiện.

In [ ]:
# 12. Asymmetric Loss
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4.0, gamma_pos=1.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, logits, targets):
        x_sigmoid = torch.sigmoid(logits)
        xs_pos = x_sigmoid
        xs_neg = 1.0 - x_sigmoid

        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1.0)

        loss = targets * torch.log(xs_pos.clamp(min=self.eps))
        loss += (1.0 - targets) * torch.log(xs_neg.clamp(min=self.eps))

        if self.gamma_neg > 0 or self.gamma_pos > 0:
            pt = xs_pos * targets + xs_neg * (1.0 - targets)
            gamma = self.gamma_pos * targets + self.gamma_neg * (1.0 - targets)
            loss *= torch.pow(1.0 - pt, gamma)

        return -loss

asl_loss = AsymmetricLoss(ASL_GAMMA_NEG, ASL_GAMMA_POS, ASL_CLIP)

In [ ]:
# 13. Tổng loss
def label_task_weight(label):
    return 1.0 if label in MAIN_LABELS else AUX_LOSS_WEIGHT


def compute_loss(outputs, batch):
    total = torch.tensor(0.0, device=DEVICE)
    total_weight = 0.0
    parts = {}

    # Binary heads
    for label in BINARY_LABELS:
        logits = outputs[label].squeeze(1)
        targets = batch['binary_targets'][label].to(DEVICE)
        mask = batch['binary_masks'][label].to(DEVICE)

        if LOSS_MODE == 'weighted_bce':
            pos_weight = torch.tensor(POS_WEIGHTS[label], dtype=torch.float32, device=DEVICE)
            per_item = F.binary_cross_entropy_with_logits(
                logits, targets, pos_weight=pos_weight, reduction='none'
            )
        elif LOSS_MODE == 'asl':
            per_item = asl_loss(logits, targets)
        else:
            raise ValueError(LOSS_MODE)

        denom = mask.sum().clamp(min=1.0)
        label_loss = (per_item * mask).sum() / denom
        w = label_task_weight(label)
        total = total + w * label_loss
        total_weight += w
        parts[label] = float(label_loss.detach().cpu())

    # Multiclass heads
    for label in MULTICLASS_LABELS:
        logits = outputs[label]
        targets = batch['multiclass_targets'][label].to(DEVICE)
        mask = batch['multiclass_masks'][label].to(DEVICE)

        per_item = F.cross_entropy(logits, targets, reduction='none')
        denom = mask.sum().clamp(min=1.0)
        label_loss = (per_item * mask).sum() / denom
        w = label_task_weight(label)
        total = total + w * label_loss
        total_weight += w
        parts[label] = float(label_loss.detach().cpu())

    total = total / max(total_weight, 1e-8)
    return total, parts

## 10. Prediction + ranking metrics

Checkpoint được chọn theo **Macro AUROC của 6 MAIN_LABELS trên development set**. Auxiliary labels không tham gia tiêu chí chọn checkpoint.

AUROC/AUPRC chỉ được tính trên raw labels `0/1`; `-1` và `NaN` không tham gia metric.

In [ ]:
# 14. Collect predictions + metrics
@torch.no_grad()
def collect_predictions(model, loader, labels=None):
    model.eval()
    labels = labels or ACTIVE_LABELS
    store = {label: {'y': [], 'p': [], 'mask': []} for label in labels}

    for batch in loader:
        images = batch['image'].to(DEVICE, non_blocking=True)
        outputs = model(images)

        for label in labels:
            probs = positive_probability_from_logits(label, outputs[label], POLICIES).detach().cpu().numpy()
            y = batch['raw_values'][label].numpy()
            mask = batch['eval_masks'][label].numpy().astype(bool)
            store[label]['p'].append(probs)
            store[label]['y'].append(y)
            store[label]['mask'].append(mask)

    result = {}
    for label in labels:
        y = np.concatenate(store[label]['y'])
        p = np.concatenate(store[label]['p'])
        mask = np.concatenate(store[label]['mask'])
        result[label] = {'y': y[mask].astype(int), 'p': p[mask]}
    return result


def safe_auroc(y, p):
    return np.nan if len(np.unique(y)) < 2 else roc_auc_score(y, p)


def safe_auprc(y, p):
    return np.nan if len(np.unique(y)) < 2 else average_precision_score(y, p)


def ranking_metrics(pred_dict, labels):
    rows = []
    for label in labels:
        y, p = pred_dict[label]['y'], pred_dict[label]['p']
        rows.append({
            'Label': label,
            'N_eval': len(y),
            'Positive': int(y.sum()),
            'Negative': int((1-y).sum()),
            'AUROC': safe_auroc(y, p),
            'AUPRC': safe_auprc(y, p),
        })
    return pd.DataFrame(rows)

In [ ]:
# 15. Train
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

history = []
best_score = -np.inf
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0

    for step, batch in enumerate(train_loader, start=1):
        images = batch['image'].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss, parts = compute_loss(outputs, batch)
        loss.backward()
        optimizer.step()

        running += float(loss.detach().cpu())
        if step % 500 == 0:
            print(f'Epoch {epoch} | step {step}/{len(train_loader)} | loss={running/step:.4f}')

    train_loss = running / max(len(train_loader), 1)

    dev_pred = collect_predictions(model, dev_loader, MAIN_LABELS)
    dev_metrics = ranking_metrics(dev_pred, MAIN_LABELS)
    macro_auroc = dev_metrics['AUROC'].mean()
    macro_auprc = dev_metrics['AUPRC'].mean()

    if CHECKPOINT_METRIC == 'macro_auroc_main':
        score = macro_auroc
    elif CHECKPOINT_METRIC == 'macro_auprc_main':
        score = macro_auprc
    else:
        raise ValueError(CHECKPOINT_METRIC)

    row = {
        'Epoch': epoch,
        'TrainLoss': train_loss,
        'MacroAUROC_MAIN': macro_auroc,
        'MacroAUPRC_MAIN': macro_auprc,
        'CheckpointScore': score,
    }
    history.append(row)

    print('\n', pd.DataFrame([row]).round(5).to_string(index=False))
    print(dev_metrics[['Label', 'AUROC', 'AUPRC']].round(4).to_string(index=False))

    if score > best_score:
        best_score = score
        best_state = deepcopy(model.state_dict())
        torch.save({
            'model_state_dict': best_state,
            'active_labels': ACTIVE_LABELS,
            'policies': POLICIES,
            'experiment_name': EXPERIMENT_NAME,
            'use_relations': CFG['use_relations'],
            'relation_sources': model.relation_sources,
            'loss_mode': LOSS_MODE,
            'pos_weights': POS_WEIGHTS,
            'nan_train_policy': NAN_TRAIN_POLICY,
            'epoch': epoch,
            'dev_macro_auroc': macro_auroc,
            'dev_macro_auprc': macro_auprc,
        }, CHECKPOINT_PATH)
        print('Saved best checkpoint ->', CHECKPOINT_PATH)

history_df = pd.DataFrame(history)
history_df

In [ ]:
# 16. Load best checkpoint
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print('Best epoch:', ckpt['epoch'])
print('Best dev Macro AUROC:', ckpt['dev_macro_auroc'])
print('Best dev Macro AUPRC:', ckpt['dev_macro_auprc'])

## 11. Chọn threshold trên development set

Threshold không ảnh hưởng AUROC, nhưng cần cho Precision/Recall/Specificity/F1. Notebook giữ mục tiêu `Recall >= 0.85`, sau đó chọn threshold có precision cao nhất trong các candidate đạt recall mục tiêu.

In [ ]:
# 17. Chọn threshold high-recall trên development set
def choose_threshold_for_recall(y_true, y_prob, target_recall=0.85):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    p = precision[:-1]
    r = recall[:-1]

    candidates = np.where(r >= target_recall)[0]
    if len(candidates) == 0:
        idx = int(np.argmax(r))
    else:
        best_local = int(np.argmax(p[candidates]))
        idx = int(candidates[best_local])

    return {
        'threshold': float(thresholds[idx]),
        'precision_dev': float(p[idx]),
        'recall_dev': float(r[idx]),
    }

best_dev_pred = collect_predictions(model, dev_loader, MAIN_LABELS)
thresholds = {}
threshold_rows = []

for label in MAIN_LABELS:
    y = best_dev_pred[label]['y']
    p = best_dev_pred[label]['p']
    info = choose_threshold_for_recall(y, p, TARGET_RECALL)
    thresholds[label] = info['threshold']
    threshold_rows.append({'Label': label, **info})

threshold_table = pd.DataFrame(threshold_rows)
threshold_table

In [ ]:
# 18. Threshold-based metrics
def threshold_metrics(pred_dict, thresholds, labels):
    rows = []
    for label in labels:
        y = pred_dict[label]['y']
        p = pred_dict[label]['p']
        t = thresholds[label]
        yhat = (p >= t).astype(int)

        if len(np.unique(y)) < 2:
            tn = fp = fn = tp = 0
            specificity = np.nan
        else:
            tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
            specificity = tn / (tn + fp) if (tn + fp) else np.nan

        rows.append({
            'Label': label,
            'Threshold': t,
            'AUROC': safe_auroc(y, p),
            'AUPRC': safe_auprc(y, p),
            'Precision': precision_score(y, yhat, zero_division=0),
            'Recall': recall_score(y, yhat, zero_division=0),
            'Specificity': specificity,
            'F1': f1_score(y, yhat, zero_division=0),
            'TP': int(tp),
            'FP': int(fp),
            'TN': int(tn),
            'FN': int(fn),
        })
    return pd.DataFrame(rows)

valid_pred = collect_predictions(model, valid_loader, MAIN_LABELS)
valid_results = threshold_metrics(valid_pred, thresholds, MAIN_LABELS)
valid_results.round(4)

In [ ]:
# 19. Macro summary + tập trung 2 nhãn cần cải thiện
macro_cols = ['AUROC', 'AUPRC', 'Precision', 'Recall', 'Specificity', 'F1']
macro_summary = valid_results[macro_cols].mean().to_frame('Macro').T
focus_results = valid_results[valid_results['Label'].isin(['Pneumonia', 'Pneumothorax'])].copy()

print('Macro summary')
print(macro_summary.round(4).to_string())
print('\nFocus labels')
print(focus_results.round(4).to_string(index=False))

## 12. Optional: đánh giá official validation sau khi đã khóa cấu hình

Cell này chỉ chạy nếu tìm thấy `valid.csv`. Không dùng official validation để chọn policy, checkpoint hoặc threshold.

In [ ]:
# 20. Official validation (optional)
if OFFICIAL_VALID_CSV.exists():
    official_valid_df = pd.read_csv(OFFICIAL_VALID_CSV)

    # Dataset cần active labels; CheXpert valid.csv thường có đủ các cột này.
    missing_official = [c for c in ['Path', *ACTIVE_LABELS] if c not in official_valid_df.columns]
    if missing_official:
        print('Bỏ qua official validation vì thiếu cột:', missing_official)
    else:
        official_valid_df = official_valid_df.copy()
        official_valid_df['PatientID'] = official_valid_df['Path'].str.extract(r'(patient\d+)', expand=False).fillna('official')

        official_ds = CheXpertRelationDataset(
            official_valid_df, ACTIVE_LABELS, POLICIES, eval_transform,
            training=False, nan_train_policy=NAN_TRAIN_POLICY
        )
        official_loader = DataLoader(
            official_ds, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=True,
        )

        official_pred = collect_predictions(model, official_loader, MAIN_LABELS)
        official_ranking = ranking_metrics(official_pred, MAIN_LABELS)
        print(official_ranking.round(4).to_string(index=False))
else:
    print('Không tìm thấy:', OFFICIAL_VALID_CSV)

## 13. Lưu kết quả

Mỗi experiment tạo file riêng để sau đó ghép bảng ablation.

In [ ]:
# 21. Save results
history_df.to_csv(OUT_DIR / f'{EXPERIMENT_NAME}_{LOSS_MODE}_history.csv', index=False)
threshold_table.to_csv(OUT_DIR / f'{EXPERIMENT_NAME}_{LOSS_MODE}_thresholds.csv', index=False)
valid_results.to_csv(OUT_DIR / f'{EXPERIMENT_NAME}_{LOSS_MODE}_internal_valid.csv', index=False)
macro_summary.to_csv(OUT_DIR / f'{EXPERIMENT_NAME}_{LOSS_MODE}_macro.csv', index=False)
relation_table.to_csv(OUT_DIR / f'{EXPERIMENT_NAME}_relation_stats.csv', index=False)

print('Saved to:', OUT_DIR.resolve())

## 14. Cách chạy ablation đúng

Chạy lần lượt bốn experiment sau, mỗi lần **restart kernel và Run All** để tránh biến/cấu hình cũ còn trong RAM:

1. `B0_main6`
2. `E1_aux_multitask`
3. `E2_relation_aware`
4. `E3_relation_aware_lsr`

Sau đó so sánh ưu tiên:

- `AUROC(Pneumonia)`
- `AUROC(Pneumothorax)`
- Macro AUROC của 6 main labels
- AUPRC của Pneumothorax vì nhãn này mất cân bằng hơn

### Cách diễn giải

- **E1 > B0**: auxiliary multi-task supervision có ích.
- **E2 > E1**: relation head mang thêm thông tin ngoài shared backbone.
- **E3 > E2**: LSR cho uncertain tiếp tục cải thiện.
- **E2/E3 không hơn E1**: không nên giữ relation head chỉ vì giả thuyết lâm sàng; shared multi-task có thể đã đủ.

### Nếu Pneumonia vẫn thấp

Thử tiếp theo theo thứ tự:

1. `AUX_LOSS_WEIGHT`: 0.1 / 0.25 / 0.35 / 0.5.
2. `RELATION_DETACH=True` để ngăn target loss làm biến dạng source heads.
3. LSR-positive chỉ cho Pneumonia, giữ Pneumothorax U-Ignore.
4. Thêm augmentation nhẹ.
5. Sau khi protocol ổn định mới thử backbone khác hoặc ensemble.

### Nếu Pneumothorax vẫn thấp

Không ép hierarchy cứng. Thử:

1. ASL thay weighted BCE.
2. Auxiliary `Pleural Effusion` + `Pleural Other` nhưng relation head tắt.
3. Phân tích false positive có liên quan `Support Devices` hay không trước khi thêm nhãn này, để tránh shortcut learning.

In [ ]:
# 22. Helper ghép kết quả ablation sau khi đã chạy nhiều experiment
# Chạy cell này khi các file *_internal_valid.csv đã tồn tại trong OUT_DIR.
rows = []
for exp_name in EXPERIMENTS:
    path = OUT_DIR / f'{exp_name}_{LOSS_MODE}_internal_valid.csv'
    if not path.exists():
        continue

    r = pd.read_csv(path)
    row = {
        'Experiment': exp_name,
        'Macro AUROC': r['AUROC'].mean(),
        'Macro AUPRC': r['AUPRC'].mean(),
    }
    for focus in ['Pneumonia', 'Pneumothorax']:
        q = r[r['Label'] == focus]
        if len(q):
            row[f'{focus} AUROC'] = float(q['AUROC'].iloc[0])
            row[f'{focus} AUPRC'] = float(q['AUPRC'].iloc[0])
    rows.append(row)

ablation_df = pd.DataFrame(rows)
if len(ablation_df):
    display(ablation_df.round(4))
else:
    print('Chưa có đủ file kết quả. Hãy chạy từng experiment trước.')